# Stacking berbobot fitur (FWLS): bobot ensemble yang berubah menurut ciri baris
**Kapan dipakai:** model yang paling andal berbeda menurut jenis baris. Contoh: model huruf
unggul di teks pendek penuh typo, model kata unggul di ulasan panjang, model tabular unggul saat
judul kosong. Bobot tetap (rata-rata, hill climbing) tidak bisa menangkap itu.

**FWLS** (Sill dkk. 2009, dipakai tim juara Netflix Prize): tebakan akhir =
Σ_model Σ_ciri v × ciri(baris) × log-peluang model. Praktisnya: regresi logistik di atas
perkalian `log-peluang model × ciri baris` (ciri = panjang teks, kolom kosong, umur, ...).

Dibandingkan (lipatan luar, jujur): tunggal, rata, stacking LR biasa, FWLS, LightGBM + ciri.
Gambar "skor per kelompok panjang" menunjukkan apakah keandalan model memang berubah.

In [ ]:
import os
import time

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

## Konfigurasi

In [ ]:
OUTPUT_FOLDER = "../../../outputs/text/stacking_fwls"
OOF_FOLDER = "../../../outputs/text/oof"
DATA_FOLDER = "../../../data_latihan/text/teks_tabular"      # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/baju_wanita_train.csv"           # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/baju_wanita_test.csv"             # GANTI saat lomba
KEY_CSV = f"{DATA_FOLDER}/baju_wanita_test_labels.csv"       # GANTI saat lomba (kunci, kalau ada)
ID_COL = "id"                                # GANTI saat lomba
TARGET = "rating"                            # GANTI saat lomba
TEXT_COLS = ["title", "review_text"]         # GANTI saat lomba
META_COLS = ["age", "positive_feedback_count"]       # GANTI: kolom angka untuk ciri baris
MODELS = ["fw_kata", "fw_huruf", "fw_judul", "fw_tabular"]   # GANTI: nama OOF
FWLS_C = 0.5
MIN_GAIN = 0.002
N_FOLDS = 5
SEED = 42


def metric(truth, pred):
    return f1_score(truth, pred.argmax(axis=1), average="macro")      # GANTI: metrik lomba


print("uji metrik:", metric(np.array([0, 1]), np.array([[0.9, 0.1], [0.2, 0.8]])))

## Baca data + ciri baris (distandarkan dengan rata-rata train)

In [ ]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
CLASSES, y = np.unique(train[TARGET].astype(str), return_inverse=True)
folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(train, y))


def meta_features(frame):
    meta = pd.DataFrame(index=frame.index)
    text = frame[TEXT_COLS].fillna("").agg(" ".join, axis=1)
    meta["log_panjang"] = np.log1p(text.str.len())
    for col in TEXT_COLS:
        meta[f"kosong_{col}"] = frame[col].isna().astype(float)
    for col in META_COLS:
        meta[f"log_{col}"] = np.log1p(frame[col].clip(lower=0))
    return meta


raw_meta_train = meta_features(train)
center = raw_meta_train.mean()
spread = raw_meta_train.std() + 1e-9
meta_train = (raw_meta_train - center) / spread
meta_test = (meta_features(test) - center) / spread
print("ciri baris:", list(meta_train.columns))

## OOF demo (HAPUS saat lomba)
Empat model yang sengaja punya kekuatan beda: kata, huruf, judul saja, tabular saja.

In [ ]:
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier

both = pd.concat([train, test], ignore_index=True)
full_text = both[TEXT_COLS].fillna("").agg(" . ".join, axis=1)
word = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100000,
                       sublinear_tf=True).fit_transform(full_text)
char = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, max_features=100000,
                       sublinear_tf=True).fit_transform(full_text)
title = TfidfVectorizer(ngram_range=(1, 2), min_df=2,
                        sublinear_tf=True).fit_transform(both["title"].fillna(""))
tabular = pd.DataFrame({"age": both["age"], "feedback": both["positive_feedback_count"]})
for col in ["division_name", "department_name", "class_name"]:
    tabular[col] = both[col].astype("category").cat.codes
tabular["panjang"] = full_text.str.len()
tabular = tabular.to_numpy(float)
OOF_FOLDER = OUTPUT_FOLDER
print("matriks demo:", word.shape, char.shape, title.shape, tabular.shape)

In [ ]:
def demo_member(matrix, model, name):
    oof = np.zeros((len(train), len(CLASSES)))
    test_pred = np.zeros((len(test), len(CLASSES)))
    for idx_fit, idx_val in folds:
        model.fit(matrix[idx_fit], y[idx_fit])
        oof[idx_val] = model.predict_proba(matrix[idx_val])
        test_pred += model.predict_proba(matrix[len(train):]) / N_FOLDS
    np.save(f"{OOF_FOLDER}/oof_{name}.npy", oof)
    np.save(f"{OOF_FOLDER}/test_{name}.npy", test_pred)


demo_member(word, SGDClassifier(loss="log_loss", alpha=2e-5, random_state=SEED), "fw_kata")
demo_member(char, SGDClassifier(loss="log_loss", alpha=2e-5, random_state=SEED), "fw_huruf")
demo_member(title, SGDClassifier(loss="log_loss", alpha=2e-5, random_state=SEED), "fw_judul")
demo_member(tabular, lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, verbose=-1,
                                        random_state=SEED), "fw_tabular")
print("OOF demo ditulis")

## Muat OOF

In [ ]:
oof_list = []
test_list = []
for name in MODELS:
    oof_list.append(np.load(f"{OOF_FOLDER}/oof_{name}.npy").reshape(len(train), -1))
    test_list.append(np.load(f"{OOF_FOLDER}/test_{name}.npy").reshape(len(test), -1))
stack = np.stack(oof_list)
test_stack = np.stack(test_list)
for number, name in enumerate(MODELS):
    print(f"{name:12s} | OOF {metric(y, stack[number]):.4f}")

## Masukan tiap penggabung
- **stacking LR**: log-peluang semua model berdampingan.
- **FWLS**: log-peluang × tiap ciri baris (+ log-peluang apa adanya).
- **LightGBM + ciri**: log-peluang + ciri baris mentah, pohon mencari interaksinya sendiri.

In [ ]:
def log_stack(members, meta):
    return np.hstack(list(np.log(np.clip(members, 1e-6, 1))))


def fwls_inputs(members, meta):
    logs = log_stack(members, meta)
    blocks = [logs]
    for col in meta.columns:
        blocks.append(logs * meta[col].to_numpy()[:, None])
    return np.hstack(blocks)


def meta_inputs(members, meta):
    return np.hstack([log_stack(members, meta), meta.to_numpy()])


print("ukuran masukan FWLS:", fwls_inputs(stack[:, :3], meta_train.iloc[:3]).shape)

In [ ]:
def make_lr():
    return LogisticRegression(C=1.0, max_iter=3000)


def make_fwls():
    return LogisticRegression(C=FWLS_C, max_iter=3000)


def make_lgbm():
    return lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=15,
                              min_child_samples=50, random_state=SEED, verbose=-1)


STACKERS = {"stacking LR": (log_stack, make_lr), "FWLS": (fwls_inputs, make_fwls),
            "LightGBM + ciri": (meta_inputs, make_lgbm)}
print("penggabung:", list(STACKERS))

## Skor jujur (lipatan luar atas baris OOF)

In [ ]:
singles = []
for number in range(len(MODELS)):
    singles.append(metric(y, stack[number]))
best_single = int(np.argmax(singles))
RESULTS = {"tunggal": (stack[best_single], test_stack[best_single]),
           "rata": (stack.mean(axis=0), test_stack.mean(axis=0))}
outer = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED + 1).split(train, y))
for name, (build, make) in STACKERS.items():
    start = time.time()
    inputs = build(stack, meta_train)
    honest = np.zeros((len(train), len(CLASSES)))
    for idx_fit, idx_val in outer:
        honest[idx_val] = make().fit(inputs[idx_fit], y[idx_fit]).predict_proba(inputs[idx_val])
    final_model = make().fit(inputs, y)
    RESULTS[name] = (honest, final_model.predict_proba(build(test_stack, meta_test)))
    print(f"{name:16s} | jujur {metric(y, honest):.4f} | detik {time.time() - start:.0f}")

## Putusan + skor per kelompok panjang teks (bahan slide)
Kalau garis model bersilangan antar kelompok, bobot tetap memang kurang dan FWLS layak dipakai.

In [ ]:
rows = []
for name in RESULTS:
    rows.append({"cara": name, "skor jujur": round(metric(y, RESULTS[name][0]), 4)})
table = pd.DataFrame(rows)
table.to_csv(f"{OUTPUT_FOLDER}/banding_cara.csv", index=False)
honest_scores = pd.Series(table["skor jujur"].to_numpy(), index=table["cara"])
CHOSEN = None
for name in RESULTS:
    if CHOSEN is None and honest_scores[name] >= honest_scores.max() - MIN_GAIN:
        CHOSEN = name
print(table.to_string(index=False), "\ndipilih:", CHOSEN)
bucket = pd.qcut(raw_meta_train["log_panjang"], 5, labels=False, duplicates="drop").to_numpy()
lines = {}
for number, name in enumerate(MODELS):
    lines[name] = stack[number]
lines["FWLS"] = RESULTS["FWLS"][0]
per_bucket = pd.DataFrame(index=np.unique(bucket))
for name, pred in lines.items():
    for group in per_bucket.index:
        per_bucket.loc[group, name] = metric(y[bucket == group], pred[bucket == group])
per_bucket.to_csv(f"{OUTPUT_FOLDER}/skor_per_panjang.csv")

In [ ]:
figure, ax = plt.subplots(figsize=(8, 4))
for name in per_bucket.columns:
    ax.plot(per_bucket.index + 1, per_bucket[name], marker="o", label=name)
ax.set_xlabel("kelompok panjang teks (1 = terpendek, 5 = terpanjang)")
ax.set_ylabel("skor OOF")
ax.legend()
figure.savefig(f"{OUTPUT_FOLDER}/skor_per_panjang.png", dpi=200, bbox_inches="tight")
plt.show()
guess = pd.DataFrame()
guess[ID_COL] = test[ID_COL]
guess[TARGET] = CLASSES[RESULTS[CHOSEN][1].argmax(axis=1)]
guess.to_csv(f"{OUTPUT_FOLDER}/tebakan_test.csv", index=False)
np.save(f"{OUTPUT_FOLDER}/tebakan_test_proba.npy", RESULTS[CHOSEN][1])

## Cek dengan kunci jawaban (HAPUS saat lomba)

In [ ]:
key = pd.read_csv(KEY_CSV)
key_by_id = pd.Series(key[TARGET].astype(str).values, index=key[ID_COL].values)
truth_test = np.searchsorted(CLASSES, test[ID_COL].map(key_by_id).to_numpy())
for name in RESULTS:
    print(f"{name:16s} | test {metric(truth_test, RESULTS[name][1]):.4f}")